In [56]:
import numpy as np
import pandas as pd

In [61]:
caption_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/caption/tn_test.csv")
caption_tn_test["data_type"] = "caption_tn"

caption_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/caption/tp_test.csv")
caption_tp_test["data_type"] = "caption_tp"


instruct_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/instruct/tn_test.csv")
instruct_tn_test["data_type"] = "instruct_tn"

instruct_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/instruct/tp_test.csv")
instruct_tp_test["data_type"] = "instruct_tp"

vqa_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/vqa/tn_test.csv")
vqa_tn_test["data_type"] = "instruct_tn"

vqa_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/vqa/tp_test.csv")
vqa_tp_test["data_type"] = "instruct_tp"

In [62]:
total_df = pd.concat([caption_tn_test, caption_tp_test, instruct_tn_test, instruct_tp_test, vqa_tn_test, vqa_tp_test])

In [64]:
total_df.index = range(len(total_df))

In [65]:
from tqdm import tqdm
all_target_words = []
failed = []
for inx, row in tqdm(total_df.iterrows()):
    try:
        res = eval(row["annotations"])
        obj_df = pd.DataFrame(res["object"])
        attr_df = pd.DataFrame(res["attribute"])
        rel_df = pd.DataFrame(res["relationship"])
        scene_df = pd.DataFrame(res["scene"])


        target_words = []
        if not obj_df.empty:
            for index, row in obj_df.iterrows():
                target_words.append({"word" : row["obj"]["name"], "type": "object", "label": "halu"})

        if not attr_df.empty:
            for index, row in attr_df.iterrows():
                target_words.append({"word" : row["attribute"]["name"], "type": "attribute", "label": "halu"})

        if not rel_df.empty:
        
            for index, row in rel_df.iterrows():
                target_words.append({"word" : row["predicate"]["name"], "type": "relationship", "label": "halu"})

        if not scene_df.empty:
            for index, row in scene_df.iterrows():
                target_words.append({"word" : row["scene"]["name"], "type": "scene", "label": "halu"})
        
        all_target_words.append(target_words)
    except:
        failed.append(inx)

26646it [00:21, 1219.62it/s]


In [66]:
len(failed)

1000

In [67]:
total_df = total_df.drop(index=failed)

In [68]:
total_df.shape

(25646, 12)

In [70]:
total_df["extracted_target_words"] = all_target_words

In [104]:
total_df.head(2)

,Unnamed: 0,image_id,prompt,hallucinated_text,source_text,source_metadata,qa_metadata,qa_ids,annotations,id,split,data_type,extracted_target_words
0,68069,2391368,Can you describe the main features of this ima...,"In this image, there are some boxes contains f...","In this image, there are some boxes contains f...","{'source': 'localized_narratives', 'id': 'sp_2...",[],[],"{'object': [], 'attribute': [], 'relationship'...",caption_4_clean,test,caption_tn,[]
1,68070,2375207,Write a detailed description of the given image.,A man is wearing a white uniform. He is wearin...,A man is wearing a white uniform. He is wearin...,"{'source': 'stanford', 'id': 'sp_30231'}",[],[],"{'object': [], 'attribute': [], 'relationship'...",caption_11_clean,test,caption_tn,[]


In [103]:
total_df["prompt"] = total_df["prompt"].apply(lambda x: x.replace("<image>", ""))

In [105]:
total_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/total_test_data_20k.csv", index=False)

In [87]:
target_test_data = total_df[total_df["data_type"].isin(["caption_tp"])]

In [88]:
target_test_data.shape

(3273, 13)

In [89]:
target_test_data.head(2)

,Unnamed: 0,image_id,prompt,hallucinated_text,source_text,source_metadata,qa_metadata,qa_ids,annotations,id,split,data_type,extracted_target_words
3273,64796,2391368,<image>Can you describe the main features of t...,"In this image, there are some boxes that conta...","In this image, there are some boxes contains f...","{'source': 'localized_narratives', 'id': 'sp_2...",[],['001006084'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_4,test,caption_tp,"[{'word': 'wicker', 'type': 'attribute', 'labe..."
3274,64797,2375207,<image>Write a detailed description of the giv...,A man is wearing a white uniform. He is wearin...,A man is wearing a white uniform. He is wearin...,"{'source': 'stanford', 'id': 'sp_30231'}",[],['001021520'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_11,test,caption_tp,"[{'word': 'plastic', 'type': 'attribute', 'lab..."


In [100]:
all_tp_words = []
all_fn_words = []
all_fp_words = []
for inx, row in tqdm(target_test_data.iterrows()):
    gt_words = [i["word"] for i in row["extracted_target_words"]]
    pred_words = [i["word"] for i in row["extracted_target_words"]]
    
    tp_words = list(set(gt_words).intersection(set(pred_words)))
    all_tp_words.extend(tp_words)
    fn_words = list(set(gt_words) - set(pred_words))
    all_fn_words.extend(fn_words)
    fp_words = list(set(pred_words) - set(gt_words))
    all_fp_words.extend(fp_words)

precision = len(all_tp_words)/(len(all_tp_words) + len(all_fp_words))
recall = len(all_tp_words)/(len(all_tp_words) + len(all_fn_words))
f1 = (2*precision*recall) / (precision + recall)

print(f"precision: {precision}")
print(f"recall: {recall}")
print(f"f1: {f1}")

3273it [00:00, 23368.41it/s]

precision: 1.0
recall: 1.0
f1: 1.0
